## Lab 2 - Part B : Inverse Kinematics

### Overview & Variables

Part B  ให้เอาคำตอบ IK ที่แก้ **ด้วยมือใน Part A** มาตรวจด้วย Robotics Toolbox แล้ว cross-check ด้วย Forward Kinematics (FK)

| Variable | What is it? | From |
|--------|---------|-------------|
| `M` | ตำแหน่งเป้าหมายใน **base frame** ของหุ่น (= `Pobj`) หน่วยเมตร `[x, y, z]` | Part A ข้อ 1: แปลง `M` จาก camera frame มาเป็น base frame ด้วย inverse ของ `T` ที่ได้รับ |
| `q_goal` | มุม/ระยะข้อต่อที่แก้ IK **ด้วยมือ** (เลือกมา 1 branch) | Part A ข้อ 4 |
| `branches` | คำตอบ IK **ทุก branch** | Part A ข้อ 4 |

**หน่วยของ `q`** ขึ้นกับชนิดข้อต่อของหุ่นที่ออกแบบใน LAB1: revolute → เรเดียน `[rad]`, prismatic → เมตร `[m]`
(หุ่นใน template นี้เป็นแบบ **RRP** → `q1, q2` = rad, `q3` = m)

**เกณฑ์ผ่าน** error ในส่วนที่ 2 และ 4 ต้องไม่เกิน 0.03 m


### 0.1 Setup

In [1]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
if hasattr(Robot, 'URDF_read'):             # older roboticstoolbox
    links, name = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)[:2]
else:                                       # roboticstoolbox 1.5 and newer
    from roboticstoolbox.models.URDF import URDFRobot
    from xacrodoc.packages import update_package_cache
    update_package_cache({'my_robot': HERE + '/my_robot'})
    links, name = URDFRobot.URDF_read(HERE + '/my_robot/robot.urdf')[:2]
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   # home configuration
env.step()

loaded my_robot 3 joints


C:\Users\User\AppData\Local\Temp\ipykernel_26704\1283573815.py:20: FutureWarning: launch(browser="notebook") is deprecated, use launch(notebook=True)
  env.launch(realtime=True, browser="notebook")


In [ ]:
# Allowable syntax (reference only - do not run as-is):
# robot.fkine(q)          # forward kinematics -> transform T ; tip position is T.t
# np.linalg.norm(a - b)   # distance between two points
# SE3(*M)                 # make a target pose from a point M = [x, y, z]
# robot.ikine_LM(SE3(*M), q0=..., mask=[1,1,1,0,0,0], joint_limits=True)   # numerical IK -> sol
# sol.q                   # the joint solution it found
# sol.success             # did it converge (True/False)

### 0.2 Cross-check your MDH model vs your URDF model

เช็คว่า URDF ตรงกับหุ่นที่คำนวณมือ (ตาราง MDH จาก LAB1) เพราะ Part A คำนวณ IK จากโมเดลที่คิดมือ

ต้องได้ `MDH vs URDF: OK` ถ้าไม่ผ่าน ให้หาว่าตาราง MDH หรือ URDF ผิด แล้วแก้ให้ตรงกันก่อนทำข้อ 1

In [2]:
# Syntax (reference only):
# RevoluteMDH(a=..., alpha=..., d=..., offset=...)        # revolute joint; joint variable is the angle
# PrismaticMDH(theta=..., a=..., alpha=..., offset=...)   # prismatic joint; joint variable is the distance
# robot_mdh.tool = SE3(x, y, z)                           # end-effector offset from the last MDH frame
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH

# edit here: your MDH table from LAB1 (use PrismaticMDH for a prismatic joint)
PI_DIV_2 = np.pi / 2
q1 = 0.00
q2 = 0.00
q3 = 0.00
L1 = 0.50
L2 = 0.10
L3 = 0.40

robot_mdh = DHRobot([
    RevoluteMDH(    a=0.0,      alpha=0.0,          d=L1                , offset=0.0),
    RevoluteMDH(    a=0.0,      alpha=PI_DIV_2,     d=0.0               , offset=PI_DIV_2),
    PrismaticMDH(   a=0.0,      alpha=PI_DIV_2,     theta=0.0           , offset=L2)
], name='my_mdh')

# edit here: set the tool transform to match your robot's end-effector
robot_mdh.tool = SE3(0.0, 0.0, L3)

print(robot_mdh)

DHRobot: my_mdh, 3 joints (RRP), modified DH parameters
┌──────┬───────┬───────────┬───────────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │    θⱼ     │    dⱼ     │
├──────┼───────┼───────────┼───────────┤
│  0.0 │  0.0° │        q1 │       0.5 │
│  0.0 │ 90.0° │  q2 + 90° │       0.0 │
│  0.0 │ 90.0° │      0.0° │  q3 + 0.1 │
└──────┴───────┴───────────┴───────────┘

┌──────┬─────────────────────────────────────┐
│ tool │ t = 0, 0, 0.4; rpy/xyz = 0°, 0°, 0° │
└──────┴─────────────────────────────────────┘



In [26]:
# Cross-check (nothing to edit): at the same q both models must put the tip at the same point
lo, hi = robot.qlim
q_tests = [np.zeros(robot.n)] + [lo + (hi - lo) * np.roll([0.25, 0.6, 0.85], k)[:robot.n] for k in range(3)]

worst = 0.0
for q in q_tests:
    p_mdh, p_urdf = robot_mdh.fkine(q).t, robot.fkine(q).t
    diff = float(np.linalg.norm(p_mdh - p_urdf))
    worst = max(worst, diff)
    print('q        :', np.round(q, 4))
    print('MDH  tip :', np.round(p_mdh, 4))
    print('URDF tip :', np.round(p_urdf, 4))
    print('pos diff :', round(diff, 4), 'm')
    print('-----------------------------------')

if worst <= 0.005:
    print('MDH vs URDF: OK')
else:
    print('MDH vs URDF: NOT MATCHING - fix your MDH table or your URDF, then run 0.1 and 0.2 again')

q        : [0. 0. 0.]
MDH  tip : [ 0.5 -0.   0.5]
URDF tip : [ 0.5 -0.   0.5]
pos diff : 0.0 m
-----------------------------------
q        : [-0.7854  0.3142  0.34  ]
MDH  tip : [ 0.5649 -0.5649  0.7596]
URDF tip : [ 0.5649 -0.5649  0.7596]
pos diff : 0.0 m
-----------------------------------
q        : [ 1.0996 -0.7854  0.24  ]
MDH  tip : [ 0.2376  0.4662 -0.0233]
URDF tip : [ 0.2376  0.4662 -0.0233]
pos diff : 0.0 m
-----------------------------------
q        : [0.3142 1.0996 0.1   ]
MDH  tip : [0.2591 0.0842 1.0346]
URDF tip : [0.2591 0.0842 1.0346]
pos diff : 0.0 m
-----------------------------------
MDH vs URDF: OK


#### Import Sympy

In [30]:
import numpy as np                                              
from spatialmath.base import trinv                              
import sympy as sp

### Symbolic FK

In [45]:
def symbolic_fk(q=None, L=(0.50, 0.10, 0.40), verbose=True):    
    """Symbolically derives Forward Kinematics (FK) for RRP     
    robot                                                       
                                                                
    using Modified DH parameters, and optionally evaluates for  
    joint values q.                                             
    """                                                         
    # 1. Define symbols                                         
    q1, q2, q3 = sp.symbols("q1 q2 q3", real=True)              
    L1, L2, L3 = sp.symbols("L1 L2 L3", real=True)              
                                                                
    # 2. Craig's Modified DH Transformation Matrix formula      
    def mdh_hand(alpha, a, theta, d):                           
        return sp.Matrix(                                       
            [                                                   
                [sp.cos(theta), -sp.sin(theta), 0, a],          
                [                                               
                    sp.sin(theta) * sp.cos(alpha),              
                    sp.cos(theta) * sp.cos(alpha),              
                    -sp.sin(alpha),                             
                    -d * sp.sin(alpha),                         
                ],                                              
                [                                               
                    sp.sin(theta) * sp.sin(alpha),              
                    sp.cos(theta) * sp.sin(alpha),              
                    sp.cos(alpha),                              
                    d * sp.cos(alpha),                          
                ],                                              
                [0, 0, 0, 1],                                   
            ]                                                   
        )                                                       
                                                                
    # 3. Individual link frames                                 
    T01 = mdh_hand(0, 0, q1, L1)                                
    T12 = mdh_hand(sp.pi / 2, 0, q2 + sp.pi / 2, 0)             
    T23 = mdh_hand(sp.pi / 2, 0, 0, q3 + L2)                    
                                                                
    # Tool offset along z of frame 3                            
    T3E = sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, L3], 
    [0, 0, 0, 1]])                                              
                                                                
    # 4. Overall transformation & Position                      
    T0E = sp.trigsimp(T01 * T12 * T23 * T3E)                    
    pos_expr = T0E[:3, 3]                                       
                                                                
    if verbose:                                                 
        print("=== Symbolic Forward Kinematics (FK) ===")       
        print("x =", pos_expr[0])                               
        print("y =", pos_expr[1])                               
        print("z =", pos_expr[2])                               
        print("========================================\n")     
                                                                
    # If numerical joint angles 'q' are passed, evaluate:       
    if q is not None:                                           
        subs = {q1: q[0], q2: q[1], q3: q[2], L1: L[0], L2:     
        L[1], L3: L[2]}                                         
        pos_num = np.array([float(val.subs(subs)) for val in    
        pos_expr])                                              
        T_num = np.array(T0E.subs(subs)).astype(np.float64)     
        return T0E, pos_expr, pos_num, T_num                    
                                                                
    return T0E, pos_expr

q_goal = np.array([0.0, 0.0, 0.0])  # edit here
T_sym, pos_sym, tip, T_num = symbolic_fk(q=q_goal,verbose=False)
print("==================================================")     
print("1. Symbolic Transformation Matrix (T_sym):")             
print("==================================================")     
sp.pprint(T_sym)                                                
                                                                
print("\n==================================================")   
print("2. Symbolic Position [x, y, z]:")                        
print("==================================================")     
print("x = ", pos_sym[0])                                       
print("y = ", pos_sym[1])                                       
print("z = ", pos_sym[2])                                           
                                                                
print("\n==================================================")   
print("3. Numerical Values:")                                   
print("==================================================")     
print(f"Tip position from symbolic FK : {np.round(tip, 4)}")    
print(f"Tip position from numerical FK: {np.round(T_num[:3, 3], 
4)}")                                                           
print(f"Difference (Error)            : {np.linalg.norm(tip - T_num[:3, 3]):.6f} m")                                          
print("==================================================")                                  

1. Symbolic Transformation Matrix (T_sym):
⎡-sin(q₂)⋅cos(q₁)  sin(q₁)   cos(q₁)⋅cos(q₂)    (L₂ + L₃ + q₃)⋅cos(q₁)⋅cos(q₂) ↪
⎢                                                                              ↪
⎢-sin(q₁)⋅sin(q₂)  -cos(q₁)  sin(q₁)⋅cos(q₂)    (L₂ + L₃ + q₃)⋅sin(q₁)⋅cos(q₂) ↪
⎢                                                                              ↪
⎢    cos(q₂)          0          sin(q₂)      L₁ + L₃⋅sin(q₂) + (L₂ + q₃)⋅sin( ↪
⎢                                                                              ↪
⎣       0             0             0                          1               ↪

↪    ⎤
↪    ⎥
↪    ⎥
↪    ⎥
↪ q₂)⎥
↪    ⎥
↪    ⎦

2. Symbolic Position [x, y, z]:
x =  (L2 + L3 + q3)*cos(q1)*cos(q2)
y =  (L2 + L3 + q3)*sin(q1)*cos(q2)
z =  L1 + L3*sin(q2) + (L2 + q3)*sin(q2)

3. Numerical Values:
Tip position from symbolic FK : [0.5 0.  0.5]
Tip position from numerical FK: [0.5 0.  0.5]
Difference (Error)            : 0.000000 m


#### Symbolic IK

In [46]:

def symbolic_ik(M=None, L=(0.50, 0.10, 0.40), verbose=True):    
    """Algebraically derives Inverse Kinematics for RRP robot   
    from FK equations                                           
                                                                
    using SymPy elimination, and optionally evaluates for target
    point M.                                                    
    """                                                         
    # 1. Define symbols                                         
    x, y, z = sp.symbols("x y z", real=True)                    
    q1, q2, q3 = sp.symbols("q1 q2 q3", real=True)              
    L1, L2, L3 = sp.symbols("L1 L2 L3", positive=True)          
                                                                
    # 2. Starting Forward Kinematics (FK) equations:            
    fx = (q3 + L2 + L3) * sp.cos(q1) * sp.cos(q2)               
    fy = (q3 + L2 + L3) * sp.sin(q1) * sp.cos(q2)               
    fz = L1 + (q3 + L2 + L3) * sp.sin(q2)                       
                                                                
    eq_x = sp.Eq(x, fx)                                         
    eq_y = sp.Eq(y, fy)                                         
    eq_z = sp.Eq(z, fz)                                         
                                                                
    #                                                                                                                 
    # Step A: Algebraic elimination of q1 and q2 to isolate q3  
    #                                                                                                                
    # In math: x^2 + y^2 + (z - L1)^2                           
    # In SymPy: sp.trigsimp() applies cos^2(q) + sin^2(q) = 1 automatically!                                              
    sum_lhs = x**2 + y**2 + (z - L1) ** 2                       
    sum_rhs = sp.trigsimp(fx**2 + fy**2 + (fz - L1) ** 2) # simplifies to (L2 + L3 + q3)**2                        
    eq_elim_q3 = sp.Eq(sum_lhs, sum_rhs)                        
                                                                
    # Solve for q3 directly with SymPy's solver:                
    q3_solutions = sp.solve(eq_elim_q3, q3)                     
    q3_expr = q3_solutions[1]  # positive physical reach solution                                                    
                                                                
    #                                                                                                               
    # Step B: Algebraic elimination to isolate q1               
    #                                                           
    #                                                     
    # Dividing fy by fx: (q3 + L2 + L3)*cos(q2) cancels out completely                                                  
    tan_q1 = sp.trigsimp(fy / fx)  # simplifies directly to tan(q1)                                                     
    q1_b1 = sp.atan2(y, x)                                      
    q1_b2 = sp.atan2(-y, -x)                                    
                                                                
    #                                                                                                               
    # Step C: Algebraic elimination to isolate q2               
    #                                                           
                                                      
    # Ground plane projection: x^2 + y^2 = (q3 + L2 + L3)^2 * cos^2(q2)                                                   
    # Height: (z - L1) = (q3 + L2 + L3) * sin(q2)               
    r_xy = sp.sqrt(x**2 + y**2)                                 
    q2_b1 = sp.atan2(z - L1, r_xy)                              
    q2_b2 = sp.atan2(z - L1, -r_xy)                             
                                                                
    sym_dict = {                                                
        "branch1": (q1_b1, q2_b1, q3_expr),                     
        "branch2": (q1_b2, q2_b2, q3_expr),                     
        "equations": {                                          
            "eq_x": eq_x,                                       
            "eq_y": eq_y,                                       
            "eq_z": eq_z,                                       
            "eq_elim_q3": eq_elim_q3,                           
            "ratio_y_x": sp.Eq(sp.Symbol("y/x"), tan_q1),       
        },                                                      
    }                                                           
                                                                
    if verbose:                                                 
        print("=== 1. Starting FK Equations ===")               
        print("eq_x :", eq_x)                                   
        print("eq_y :", eq_y)                                   
        print("eq_z :", eq_z)                                   
        print("\n=== 2. Algebraic Elimination for q3 ===")      
        print("x^2 + y^2 + (z - L1)^2 simplifies to:", sum_rhs) 
        print("Eliminated equation   :", eq_elim_q3)            
        print("sp.solve result for q3:", q3_expr)               
        print("\n=== 3. Algebraic Elimination for q1 ===")      
        print("fy / fx simplifies to :", tan_q1)                
        print("Branch 1 q1           :", q1_b1)                 
        print("Branch 2 q1           :", q1_b2)                 
        print("\n=== 4. Algebraic Elimination for q2 ===")      
        print("Branch 1 q2           :", q2_b1)                 
        print("Branch 2 q2           :", q2_b2)                 
        print("=========================================\n")    
                                                                
    # If numerical target M is passed, substitute and return numpy arrays:                                               
    if M is not None:                                           
        subs = {x: M[0], y: M[1], z: M[2], L1: L[0], L2: L[1],  
        L3: L[2]}                                               
        b1_num = np.array(                                      
            [float(expr.subs(subs)) for expr in                 
            sym_dict["branch1"]]                                
        )                                                       
        b2_num = np.array(                                      
            [float(expr.subs(subs)) for expr in                 
            sym_dict["branch2"]]                                
        )                                                       
        return sym_dict, b1_num, b2_num                         
                                                                
    return sym_dict     

          

### 1. target M (from Part A item 1) and q (from Part A item 4)

In [47]:
# M = [ 0.0347,  0.0563,  0.9768 ]
# M => Cam_M_Object
#     [   0.9955   -0.0949    0.0003   -0.4325 ]
# Cam_T_Base = [  -0.0947   -0.9942   -0.0518   -0.0507 ]
#     [   0.0052    0.0515   -0.9987    1.9335 ]
#     [   0.0000    0.0000    0.0000    1.0000 ]
      
# A 4x4 homogeneous transformation matrix 
Cam_T_Object = np.array(                                                   
    [                                                           
        [0.0,0.0,0.0,0.0347],                                  
        [0.0,0.0,0.0,0.0563],                                   
        [0.0,0.0,0.0,0.9768],                                   
        [0.0, 0.0, 0.0, 1.0],                                   
    ]                                                           
)                      
Cam_T_Base = np.array(                                                   
    [                                                           
        [0.9955, -0.0949, 0.0003, -0.4325],                                  
        [-0.0947, -0.9942, -0.0518, -0.0507],                                   
        [0.0052, 0.0515, -0.9987, 1.9335],                                   
        [0.0, 0.0, 0.0, 1.0],                                   
    ]                                                           
)                                                               

# Calculate the inverse transformation                          
Cam_T_Base_inv = np.round(trinv(Cam_T_Base),4)                                                                                                        
print("T_inv:\n", Cam_T_Base_inv)  
print("-----------------------------\n")  

Base_T_Object = np.round(Cam_T_Base_inv @ Cam_T_Object,4)
print("Base_T_Object:\n", Base_T_Object)
print("-----------------------------\n")  

M = np.array([0.45, -0.2, 0.9501])       # edit here

sym_dict, branch1, branch2 = symbolic_ik(M=M, verbose=False) 

q_goal = branch1                                                
# q_goal = np.array([0.0, 0.0, 0.0])  # edit here
                                                                
print("q_goal:", np.round(q_goal, 4))     

if np.allclose(M, 0) or np.allclose(q_goal, 0):
    print('fill in M and q_goal first, then run again')
else:
    marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))
    marker.T = SE3(*M)
    env.add(marker)
    env.step()
    print('target shown at', np.round(M, 4))

T_inv:
 [[ 9.9550e-01 -9.4700e-02  5.2000e-03  4.1570e-01]
 [-9.4900e-02 -9.9420e-01  5.1500e-02 -1.9100e-01]
 [ 3.0000e-04 -5.1800e-02 -9.9870e-01  1.9285e+00]
 [ 0.0000e+00  0.0000e+00  0.0000e+00  1.0000e+00]]
-----------------------------

Base_T_Object:
 [[ 0.      0.      0.      0.45  ]
 [ 0.      0.      0.     -0.2   ]
 [ 0.      0.      0.      0.9501]
 [ 0.      0.      0.      1.    ]]
-----------------------------

q_goal: [-0.4182  0.7405  0.1672]
target shown at [ 0.45   -0.2     0.9501]


In [48]:
# move the robot to q_goal  (tip should reach the ball)
robot.q = q_goal
env.step()

### 2. cross-check with FK

In [49]:

q = q_goal                                                      
                                                                
# Call your symbolic FK with q_goal                             
_, _, tip, _ = symbolic_fk(q=q, verbose=False)                  
dist = np.linalg.norm(tip - M) 

# tip = None    # edit here
# dist = None   # edit here

if dist is None:
    print('fill in tip and dist above')
else:
    print('tip:', np.round(tip, 4), '| M:', np.round(M, 4), '| err:', round(float(dist), 4))

tip: [ 0.45   -0.2     0.9501] | M: [ 0.45   -0.2     0.9501] | err: 0.0


### 3. let the toolbox solve IK (ikine), compare with q_goal

In [50]:
sol = robot.ikine_LM(SE3(*M), q0=q_goal, mask=[1, 1, 1, 0, 0, 0], joint_limits=True)

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
    print('hand q   :', np.round(q_goal, 4))

toolbox q: [-0.4182  0.7405  0.1672] | ok True
hand q   : [-0.4182  0.7405  0.1672]


### 4. check every branch

In [51]:
branches = [
    branch1,    # edit here : branch 1
    branch2     # edit here : branch 2
]

for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4))

branch 1 [-0.4182  0.7405  0.1672] -> err 0.0
branch 2 [2.7234 2.4011 0.1672] -> err 0.0
